# NF-v3 temporal graph construction

This notebook is intentionally thin. The tested graph builder lives in `code/python/scripts/build_nfv3_graphs.py`; this notebook only mounts Drive, configures paths, runs a small smoke build, and inspects its audit.

Run the smoke build first. Do not run the full build until its generated schema, mappings, provenance, and audit have been reviewed.

In [ ]:
from pathlib import Path
import subprocess

REPO_ROOT = Path('/content/temporalgnn-nids')
REPO_URL = 'https://github.com/tatipar/temporalgnn-nids.git'
BRANCH = 'feat/fair-retrain-clean'

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)], check=True)

!pip install -q torch-geometric

from google.colab import drive
drive.mount('/content/drive')

import json

PROJECT_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain')
CORRECTED_ROOT = PROJECT_ROOT / 'corrected_data' / 'infiltration_v1'
CORRECTED_CSV = CORRECTED_ROOT / 'nfv3_corrected.csv'
CORRECTED_MANIFEST = CORRECTED_ROOT / 'nfv3_corrected.manifest.json'

assert REPO_ROOT.is_dir(), f'Repository not found: {REPO_ROOT}'
assert CORRECTED_CSV.is_file(), f'Corrected CSV not found: {CORRECTED_CSV}'
assert CORRECTED_MANIFEST.is_file(), f'Corrected manifest not found: {CORRECTED_MANIFEST}'

GRAPH_VERSION = 'infiltration_v1_w30'
PREFLIGHT_ROOT = PROJECT_ROOT / 'graphs' / f'{GRAPH_VERSION}_preflight'
SMOKE_ROOT = PROJECT_ROOT / 'graphs' / f'{GRAPH_VERSION}_smoke'
FULL_ROOT = PROJECT_ROOT / 'graphs' / GRAPH_VERSION
PROFILES = ('nfv3_extended', 'portable_core')

print(f'Corrected CSV: {CORRECTED_CSV}')
print(f'Preflight output: {PREFLIGHT_ROOT}')
print(f'Smoke output: {SMOKE_ROOT}')
print(f'Full output: {FULL_ROOT}')

In [ ]:
preflight_command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(PREFLIGHT_ROOT),
    '--chunksize', '250000',
    '--preflight-only',
    '--overwrite',
]
for profile in PROFILES:
    preflight_command += ['--profile', profile]

print(' '.join(preflight_command))
subprocess.run(preflight_command, check=True)

preflight = json.loads((PREFLIGHT_ROOT / 'feature_preflight.json').read_text())
print('Preflight status:', preflight['status'])
print('Invalid ports:', preflight['invalid_port_rows'])
print('Invalid protocols:', preflight['invalid_protocol_rows'])
print('Invalid numeric rows:', preflight['invalid_numeric_rows_by_profile'])
print('Port categories:', preflight['port_category_counts'])
print('Port zero by protocol:', preflight['port_zero_by_protocol'])
print('Top other privileged ports:', preflight['other_privileged_top_ports'])
print('Top other high ports:', preflight['other_high_top_ports'])

In [ ]:
command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(SMOKE_ROOT),
    '--chunksize', '250000',
    '--max-windows', '10',
    '--overwrite',
]
for profile in PROFILES:
    command += ['--profile', profile]

print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
audit = json.loads((SMOKE_ROOT / 'graph_audit.json').read_text())
configuration = json.loads((SMOKE_ROOT / 'build_configuration.json').read_text())

print('Audit status:', audit['status'])
print('Day-1 cutoffs:', configuration['day1_cutoffs'])
print('Profile schema hashes:')
for name, digest in configuration['profiles'].items():
    print(f'- {name}: {digest}')

for day_name in ('day1', 'day2'):
    mapping = json.loads((SMOKE_ROOT / 'mappings' / f'{day_name}_ip_to_id.json').read_text())
    print(f"{day_name}: {mapping['entries']} mapped IPs; policy={mapping['creation_policy']}")

In [ ]:
import pandas as pd
import torch

sample_provenance = next((SMOKE_ROOT / 'provenance' / 'day1').glob('graph_*.csv'))
sample_graph = SMOKE_ROOT / 'nfv3_extended' / 'train' / f'{sample_provenance.stem}.pt'
provenance = pd.read_csv(sample_provenance)
graph = torch.load(sample_graph, weights_only=False)

print('Graph:', sample_graph.name)
print('Nodes:', graph.num_nodes)
print('Edges:', graph.edge_index.shape[1])
print('Edge feature shape:', tuple(graph.edge_attr.shape))
print('Decision time (UTC epoch ms):', graph.timestamp)
display(provenance.head())

## Full build

Run this cell only after reviewing the smoke-build output. Use a new versioned output directory instead of overwriting a reviewed graph collection. If Colab disconnects, rerun the same command with `--resume`.

In [ ]:
# full_command = [
#     'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
#     '--input-csv', str(CORRECTED_CSV),
#     '--corrected-manifest', str(CORRECTED_MANIFEST),
#     '--output-root', str(FULL_ROOT),
#     '--chunksize', '250000',
# ]
# for profile in PROFILES:
#     full_command += ['--profile', profile]
# subprocess.run(full_command, check=True)

# To resume an interrupted full build, append '--resume' to full_command and run it again.